##### RAG pipeline using Hugging Face embeddings, FAISS, and LangChain, covering both text and PDF workflows

In [28]:

from dotenv import load_dotenv

In [29]:

load_dotenv()

True

In [30]:

import os

os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [31]:

from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1980.56it/s]


### Working with PDF

In [32]:

from langchain_community.document_loaders import PyPDFLoader

In [33]:

FILE_PATH="AI and DATA SCIENCE.pdf"

In [34]:

loader=PyPDFLoader(FILE_PATH)

In [35]:

len(loader.load())

11

In [36]:

pages=loader.load()
pages

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2025-07-07T05:04:36+05:30', 'author': 'GanapatiRaju Nadimpalli', 'moddate': '2025-07-07T05:04:36+05:30', 'source': 'AI and DATA SCIENCE.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='AI & DATA SCIENCE USING PYTHON \nIntroduction to AI ML DS , Gen AI and Agentic AI (DEMO) \n• Relationship between AI, ML, DL, NLP, Data Science, Gen AI, Agentic AI \n• Applications for data science \n• Why Python for Data Science \n• AI Tools: Claude/Perplexity/Co Pilot/ChatGPT/Deep Seek \n \nPython Programming \nPython Overview  \n• Installation of Anaconda Python,  \n• Python Features \n• Variables, Operators, Data Types \n• Conditions  \n• Loops \n• Functions \n• Modules, Packages \n• String object, Exercises \n• List object, Exercises, Case study \n• Tuple object, Exercises \n• Dictionary object, Exercises, Case Study \n• Set, Frozenset  \n• Compre

In [38]:

pages = []

async for page in loader.alazy_load():
    pages.append(page)

Step-by-step:

    loader.alazy_load()
        → Returns an async generator of Document objects

    async for
        → Asynchronously iterates over documents as they are loaded

    Each page is a:

       Document( page_content="...",
                metadata={...}
               )
    All pages are collected into a list: pages

| Method         | Behavior                           |
| -------------- | ---------------------------------- |
| `load()`       | Blocking, loads everything at once |
| `lazy_load()`  | Generator (sync)                   |
| `alazy_load()` | **Async generator (non-blocking)** |



Efficient for large PDFs / web pages

Lower memory pressure

Works well in async pipelines


In [39]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [42]:

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,#hyperparameter
    chunk_overlap=50 #hyperparemeter
)

In [43]:

split_docs = splitter.split_documents(pages)

In [44]:

len(split_docs)

24

In [45]:
import faiss

from langchain_community.vectorstores import FAISS

from langchain_community.docstore.in_memory import InMemoryDocstore


In [46]:

index=faiss.IndexFlatIP(384)

vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [47]:
vector_store.add_documents(documents=split_docs)


['8a8c20f2-a2b0-4622-821a-12b4dc5f16e6',
 '5ff93a66-87bb-4820-a82b-dabc583f5a72',
 '0bd283be-64f9-4e67-b075-22149242dbf5',
 'b35673ae-ff09-44b4-83d6-3b38636f7b95',
 'd2c8b191-6ae4-4d74-aae5-a302bd2996b9',
 '94cdafae-2615-4e4b-a99c-3c2720b10e6e',
 'bcc7d3e7-919b-45a9-a30b-cb05caf49d89',
 '263a2af6-a0de-4251-b91e-3ba42ed57a55',
 '63476daf-9345-4d78-ba95-032a16df11f7',
 '613e4e36-5ab8-4fca-a330-ee3eb6986ed7',
 'cb78cede-a294-4938-8c8c-01fb003fce7c',
 '6ea63785-4b02-418d-8059-8cc94eaa65c1',
 '95a253ff-3f61-4245-b2ee-3e1092d9a56f',
 'aa883d8b-08d3-4b05-843c-3fb6375642b7',
 '90cd40ff-6f1a-4af2-bdcb-48c362b9edd5',
 '7d344a38-cd98-4e97-a075-e26f30c5773e',
 '58b9f123-de55-447b-a46f-49fa25bbb389',
 '4e343e52-a001-4b5b-95aa-b992c1d973e8',
 '46204338-43c8-49ba-9551-cdccff1d4d09',
 '8362d1b5-efa7-4ca7-92be-65b2a7a79387',
 'b4018d90-acc1-448a-9c31-e2409523220a',
 'f508fdb6-5ad0-4f13-9591-c5833a66bd24',
 'c9c52b2d-0fea-4bcb-9a34-affbc824db2b',
 '57e7cd10-19b1-4dbd-a2c0-4c9f832a2ea7']

In [48]:

retriever=vector_store.as_retriever(
    search_kwargs={"k": 4} 
)

In [49]:
# Assuming retriever output is stored in 'retrieved_docs'

retrieved_docs = retriever.invoke("Give me syllabus for Machine Learning ?")

print(f"\n Retrieved {len(retrieved_docs)} relevant chunks:\n")

for i, doc in enumerate(retrieved_docs, start=1):

    print(f"📘 Document {i}")
  
    print(doc.page_content.strip())
    print("\n")


 Retrieved 4 relevant chunks:

📘 Document 1
AI & DATA SCIENCE USING PYTHON 
Introduction to AI ML DS , Gen AI and Agentic AI (DEMO) 
• Relationship between AI, ML, DL, NLP, Data Science, Gen AI, Agentic AI 
• Applications for data science 
• Why Python for Data Science 
• AI Tools: Claude/Perplexity/Co Pilot/ChatGPT/Deep Seek 
 
Python Programming 
Python Overview  
• Installation of Anaconda Python,  
• Python Features 
• Variables, Operators, Data Types 
• Conditions  
• Loops 
• Functions 
• Modules, Packages


📘 Document 2
Classification 
• Introduction to Machine Learning 
• Naïve Bayes classifier 
• Decision Tree classifier


📘 Document 3
o Unsupervised learning based 
• Case Studies on Data Preprocessing techniques and comparative analysis of various 
techniques 
 
Regression Analysis 
• Regression, Linear regression 
• Linear regression, Multiple Regression 
• Ridge Regression, Lasso Regression  
• Explanation of statistics 
• Evaluation metrics (R-Squre, Adj R-Squre, MSE, RMS